# 00d. PyTorch Core Mechanisms (80/20 Mastery Guide)

Everything you learned in **Python OOP (`class`, `super().__init__()`, `__call__`, `__len__`, `__getitem__`)** and **NumPy (`shape`, `@`, broadcasting)** snaps directly together inside **PyTorch**.

Think of PyTorch as **NumPy + 2 Superpowers**:
1. **GPU / Apple Silicon acceleration** (if available)
2. **Autograd** — automatic calculus (`loss.backward()`) so you never have to hand-derive gradients for 50-layer networks!

### What You Will Master (80/20 Rule):
1. `torch.Tensor` vs `np.ndarray` (`float32` vs `int64` / `long`)
2. Autograd (`requires_grad=True`, `.backward()`, `.grad`, `torch.no_grad()`, `.detach()`)
3. Building Models with `nn.Module` (`super().__init__()` + `forward()`)
4. Essential Layers (`Linear`, `Conv2d`, `LSTM`, `Embedding`) & exact shapes
5. `Dataset` & `DataLoader` (using `__len__` and `__getitem__`)
6. Loss Functions & Optimizers (`CrossEntropyLoss` expects **raw logits**, NOT softmax!)
7. The Universal 5-Step Training Loop
8. `model.train()` vs `model.eval()` + Saving/Loading `state_dict()`

## 1. Tensors vs. NumPy Arrays (`float32` vs `long`)

A `torch.Tensor` looks almost identical to a NumPy array (`.shape`, `.dtype`, `@`, broadcasting), with two critical differences:
1. PyTorch prefers **`torch.float32`** (not `float64`) for features and model weights.
2. Class labels (`targets`) and token IDs (`Embedding` inputs) **MUST** be **`torch.int64` (called `torch.long`)**, or `CrossEntropyLoss` crashes!
3. Choose the best compute `device` automatically (`cuda` → `mps` on Mac → `cpu`).

In [ ]:
import numpy as np
import torch

# Select device cleanly
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("Using device:", device)

# Convert NumPy <-> PyTorch
x_np = np.array([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]], dtype=np.float32)
x_pt = torch.from_numpy(x_np)            # shares memory on CPU!
y_pt = torch.tensor([0, 1], dtype=torch.long)

print("Tensor shape:", x_pt.shape, "| dtype:", x_pt.dtype)
print("Labels shape:", y_pt.shape, "| dtype:", y_pt.dtype)
print("Back to NumPy:\n", x_pt.numpy())

## 2. Autograd: How PyTorch Computes Derivatives Automatically

When you set `requires_grad=True` on a tensor `w`, PyTorch silently records every math operation performed on `w` into a **computation graph**.

* Calling **`loss.backward()`** walks backward through that graph using the chain rule and writes $\frac{\partial \text{loss}}{\partial w}$ into `w.grad`.
* **CRITICAL TRAP:** PyTorch **adds (accumulates)** new gradients onto old `w.grad`! If you don't reset `w.grad.zero_()` before the next step, gradients pile up across steps!
* **`with torch.no_grad():`** pauses graph recording (used during validation/testing so you save memory and speed).
* **`.detach()`** disconnects a tensor from the graph so you can inspect `.numpy()` safely.

In [ ]:
# Let's differentiate f(w) = (w * x - y)^2 where x=2, y=10, starting w=3.0
# By hand: f(3) = (6 - 10)^2 = 16; df/dw = 2*(w*x - y)*x = 2*(-4)*2 = -16
w = torch.tensor(3.0, requires_grad=True)
x = torch.tensor(2.0)
y = torch.tensor(10.0)

loss = (w * x - y) ** 2
loss.backward()  # Computes df/dw automatically!

print("Loss value:", loss.item())        # .item() extracts a single Python scalar
print("Computed df/dw in w.grad:", w.grad.item())  # -> -16.0!

# Reset gradient before next computation!
w.grad.zero_()
print("After zero_():", w.grad.item())

# Inference / weight update without tracking gradients:
with torch.no_grad():
    updated_w = w - 0.1 * 1.0
print("updated_w tracks grad?", updated_w.requires_grad)  # False

## 3. Building Any Model: Subclassing `nn.Module`

Look familiar from `00a`? Every PyTorch model follows this exact 2-method template:
1. **`__init__(self, ...)`**:
   * **MUST call `super().__init__()` first!** (Otherwise `model.parameters()` returns empty and nothing learns!)
   * Define your layers (`self.fc1 = nn.Linear(...)`). PyTorch automatically registers every `nn.Module` saved onto `self.`.
2. **`forward(self, x)`**:
   * Defines how input `x` flows through those layers.
   * **Calling rule:** Write **`out = model(x)`** (which triggers `nn.Module.__call__`, runs PyTorch hooks, then calls your `forward(x)`), NOT `model.forward(x)` directly.

In [ ]:
import torch.nn as nn

class TinyMLP(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int, num_classes: int):
        super().__init__()  # MANDATORY! Registers parameters inside PyTorch
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)  # returns raw scores (logits), NOT softmax!

model = TinyMLP(in_dim=8, hidden_dim=16, num_classes=3)
print(model)
total_params = sum(p.numel() for p in model.parameters())
print("Trainable parameters:", total_params)

dummy_x = torch.randn(4, 8)   # batch of 4 samples, 8 features
logits = model(dummy_x)       # calls __call__ -> forward()
print("Logits shape:", logits.shape)  # (4, 3)

## 4. Essential PyTorch Layers & Their Exact Input/Output Shapes

Almost every neural architecture in your roadmap comes down to these 4 layers:

| Layer | Purpose | Expected Input Shape | Output Shape |
|---|---|---|---|
| `nn.Linear(D_in, D_out)` | Dense / MLP / Classifier head | `(B, D_in)` or `(B, T, D_in)` | `(B, D_out)` or `(B, T, D_out)` |
| `nn.Conv2d(C_in, C_out, kernel_size=3, padding=1)` | 2D Spectrogram / Image CNN | `(B, C_in, H, W)` | `(B, C_out, H, W)` |
| `nn.Embedding(vocab_size, emb_dim)` | Token IDs (SMILES / peptides) → vectors | `(B, T)` (**`torch.long`!**) | `(B, T, emb_dim)` |
| `nn.LSTM(emb_dim, hidden, batch_first=True)` | Sequence encoder | `(B, T, emb_dim)` | `out: (B, T, hidden)`, `(h_n, c_n)` |

> **Critical Flag:** Always pass **`batch_first=True`** to `nn.LSTM` and `nn.GRU`! Without it, PyTorch puts `Time` before `Batch` `(T, B, D)`, which silently scrambles your batch!

In [ ]:
# 1. Conv2d shape check (e.g., 1-channel Mel-spectrogram of 64x128)
conv = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
spec = torch.randn(4, 1, 64, 128)
print("Conv2d output:", conv(spec).shape)  # (4, 16, 64, 128)

# 2. Embedding + LSTM shape check (e.g., 4 peptides/SMILES of length 10)
emb = nn.Embedding(num_embeddings=25, embedding_dim=32)
lstm = nn.LSTM(input_size=32, hidden_size=64, batch_first=True, bidirectional=True)

token_ids = torch.randint(0, 25, (4, 10), dtype=torch.long)
vectors = emb(token_ids)                   # (4, 10, 32)
lstm_out, (h_n, c_n) = lstm(vectors)       # BiLSTM doubles hidden_size -> 128!
print("Embedding out:", vectors.shape)
print("BiLSTM all steps:", lstm_out.shape) # (4, 10, 128)
print("Last step features:", lstm_out[:, -1, :].shape)  # (4, 128)

## 5. `Dataset` & `DataLoader`: Feeding Batches Cleanly

Remember `__len__` and `__getitem__` from Notebook `00a`? PyTorch's `Dataset` literally uses those two dunder methods!
* **`Dataset`:** Defines how to fetch **1 sample** (`__getitem__(idx)`) and total count (`__len__`).
* **`DataLoader`:** Wraps your `Dataset`, automatically gathers `batch_size` samples into a batch tensor, and shuffles every epoch!

In [ ]:
from torch.utils.data import Dataset, DataLoader

class TabularDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self) -> int:
        return len(self.y)

    def __getitem__(self, idx: int):
        return self.X[idx], self.y[idx]

# Synthetic dataset: 100 samples, 8 features, 2 classes
X_np = np.random.randn(100, 8).astype(np.float32)
y_np = (X_np[:, 0] + X_np[:, 1] > 0).astype(np.int64)

ds = TabularDataset(X_np, y_np)
loader = DataLoader(ds, batch_size=16, shuffle=True)

batch_x, batch_y = next(iter(loader))  # grab first mini-batch
print("Total batches per epoch:", len(loader))
print("One batch X shape:", batch_x.shape, "| y shape:", batch_y.shape)

## 6. Loss Functions & Optimizers (Important Softmax Rule!)

* **Classification (`nn.CrossEntropyLoss()`):**
  Already applies `LogSoftmax + NLLLoss` internally!
  **NEVER put `nn.Softmax()` at the end of your model if using `CrossEntropyLoss`** — pass raw `logits` directly!
* **Binary Classification (`nn.BCEWithLogitsLoss()`):**
  Takes raw 1D `float32` logits and `float32` targets `(0.0 or 1.0)`.
* **Regression / IC50 (`nn.MSELoss()`):**
  Mean squared error on continuous targets.
* **Optimizer (`torch.optim.AdamW(model.parameters(), lr=1e-3)`):**
  Receives `model.parameters()` and knows how to update every weight using `.grad`.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-2, weight_decay=1e-4)

raw_logits = torch.tensor([[2.5, 0.2, -1.0],
                           [0.1, 1.9, 0.3]])   # raw scores, NO softmax!
true_labels = torch.tensor([0, 1], dtype=torch.long)

loss = criterion(raw_logits, true_labels)
print("CrossEntropyLoss:", loss.item())

## 7. The Universal 5-Step Training Loop + `train()` vs `eval()`

Every PyTorch training script—from tiny MLPs to GPT—runs the **exact same 5 lines** inside each batch loop:
1. `optimizer.zero_grad()` — wipe old gradients from the previous batch
2. `logits = model(xb)` — forward pass
3. `loss = criterion(logits, yb)` — compute scalar loss
4. `loss.backward()` — autograd computes gradients for every parameter
5. `optimizer.step()` — optimizer nudges weights downhill

### `model.train()` vs `model.eval()`
* **`model.train()`:** Enables `Dropout` and updates `BatchNorm` running statistics.
* **`model.eval()` + `with torch.no_grad():`:** Disables `Dropout`, freezes `BatchNorm`, and turns off autograd for clean, deterministic evaluation!

In [ ]:
torch.manual_seed(42)

# Clean end-to-end 2-class classification experiment
N, D = 200, 8
X_all = torch.randn(N, D)
# Non-linear rule: class 1 if x0^2 + x1^2 > 1.5 else 0
y_all = ((X_all[:, 0] ** 2 + X_all[:, 1] ** 2) > 1.5).long()

train_ds = TabularDataset(X_all[:160].numpy(), y_all[:160].numpy())
val_ds = TabularDataset(X_all[160:].numpy(), y_all[160:].numpy())
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)

clf = TinyMLP(in_dim=D, hidden_dim=32, num_classes=2)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(clf.parameters(), lr=1e-2)

for epoch in range(1, 21):
    # --- TRAIN ---
    clf.train()
    train_loss = 0.0
    for xb, yb in train_loader:
        optimizer.zero_grad()       # 1. reset grads
        logits = clf(xb)            # 2. forward
        loss = criterion(logits, yb)# 3. loss
        loss.backward()             # 4. backward
        optimizer.step()            # 5. update weights
        train_loss += loss.item()

    # --- VALIDATE ---
    if epoch in (1, 5, 10, 20):
        clf.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                preds = clf(xb).argmax(dim=1)
                correct += (preds == yb).sum().item()
                total += len(yb)
        print(f"Epoch {epoch:2d} | train_loss={train_loss/len(train_loader):.4f} | val_acc={correct/total:.2%}")

## 8. Saving & Loading Model Weights (`state_dict`)

Best practice in PyTorch: save `model.state_dict()` (just the weight dictionary), NOT the entire Python object! That way your checkpoint never breaks when you rename files or folders later.

In [ ]:
import io

# Save weights to an in-memory buffer (or pass a file path like "best_model.pt")
buffer = io.BytesIO()
torch.save(clf.state_dict(), buffer)
buffer.seek(0)

# To reload later: create fresh model architecture first, then load weights
reloaded = TinyMLP(in_dim=D, hidden_dim=32, num_classes=2)
reloaded.load_state_dict(torch.load(buffer, weights_only=True))
reloaded.eval()

test_sample = X_all[160:163]
with torch.no_grad():
    assert torch.allclose(clf(test_sample), reloaded(test_sample))
print("Weights saved & reloaded cleanly with exact match!")

---
## Practice Challenges (Try Without Peeking!)

1. Add a 2nd hidden layer to `TinyMLP` (`in_dim -> 32 -> 16 -> num_classes`) and verify `total_params` increases cleanly.
2. Write a minimal `SequenceClassifier(nn.Module)` that takes `(batch, seq_len)` token IDs → `nn.Embedding(20, 16)` → `nn.GRU(16, 32, batch_first=True)` → takes the last timestep `out[:, -1, :]` → `nn.Linear(32, 2)`. Test it on `torch.randint(0, 20, (5, 12))`.